#**Reto computacional II:** El virial del sistema solar.

<div align="left">


## *Mecánica Celeste (2024-2)*
### Juliana Ruiz Montoya - 1007435437

El reto consiste en calcular el promedio de la energía cinética y potencial del sistema solar.  Para ello use como guía la teoría vista en clase y el notebook de la clase:

* Calcule la energía cinética promedio a lo largo de un lapso de tiempo que sea igual a 2 veces el período de traslación del planeta más lejano.
* Calcule la energía potencial promedio en el mismo intervalo.
* Calcule la energía total en el mismo intervalo.
* Verifique si se cumple el teorema del virial: E + K = 0 o 2K + U = 0.
* Use distintos grupos de planetas: Sol, Júpiter y Saturno (como usamos en clase), después agrege a Urano y Neptuno y compare.

Primero, se instala pymcel para descargar datos necesarios del sistema solar, además se importan las librerías necesarias.

In [1]:
!pip install -Uq pymcel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 20.7 MB/s eta 0:00:00


In [2]:
import pymcel as pc
import spiceypy as spy
import numpy as np

Paquete pymcel cargado. Versión: 0.6.10


In [3]:
pc.descarga_kernels()

In [4]:
spy.furnsh('pymcel/data/gm_de431.tpc')


## **Sistema de cuerpos: Sol-Júpiter-Saturno**

El planeta con el periodo orbital (traslación) más lejano en el sistema solar es Neptuno, el cual corresponde a:

$$ T_N = 164\ years + 280\ days = (59860 + 280)\ days = 60140\ days $$

Esto nos servirá para establecer el stop de la época en que calcularemos los datos, así los calculos de la energía del sistema se hará en un lapso de 2 veces el planeta con periodo orbital más largo.


In [5]:
T_N = 60140 # días
T_N2 = 2*T_N # días
t = T_N2/365
print("El periodo orbital de Neptuno dos veces es: ",t, "lo cual es aproximadamente:", np.round(t,0))

El periodo orbital de Neptuno dos veces es:  329.5342465753425 lo cual es aproximadamente: 330.0


Se calcula la **energía cinética** promedio del sistema:

$$ K = \frac {1}{2} mv^2 $$

Para esto creamos un ciclo for el cual nos extrae la información de vectores de estado de los planetas y sus masas. Luego, creamos la variable $K$ la cual almacenará los valores de la energía cinética de los cuerpos, por último, se hace el promedio de la energía cinética en el sistema.

In [13]:
K_total = 0

for id in '10', '5', '6':
  # Extraemos los datos
  m = pc.consulta_propiedad(id = id, propiedad = 'masa')
  Xs, ts, Xs_df = pc.consulta_horizons(
        id = id,
        location = '@0',
        epochs =dict(start = '2024-09-23', stop = '2353-09-23', step = '365d')
        )

  # Sacamos las posiciones y velocidades
  Xs = np.array(Xs_df)
  rs = Xs[:,:3]
  vs = Xs[:,3:]

  # Energía cinetica de un cada cuerpo
  K = 1/2 * m * np.linalg.norm(vs, axis=1) ** 2

  K_total += K

K_mean = K_total.mean()
print("La energía cinética promedio del sistema es:", K_mean)

La energía cinética promedio del sistema es: 1.882113321004956e+35


La **energía potencial** se define así:

$$ U = \frac{- GMm}{\vec{r}} $$

Para calcularla hay que guardar la información de todas las posiciones de los planetas, se crean listas para almacenar los valores de vectores posición y otra para almacenar las masas.

In [7]:
G = pc.constantes.G
G

6.6743e-11

In [8]:
Rs = []
ms = []
for id in '10', '5', '6':
  # Extraemos los datos
  m = pc.consulta_propiedad(id=id,propiedad='masa')
  ms += [m]

  Xs,ts,Xs_df = pc.consulta_horizons(
      id=id,
      location='@0',
      epochs=dict(start='2024-09-23',stop='2353-09-23',step='365d')
  )

  # Sacamos posiciones y velocidades
  Xs = np.array(Xs_df)
  rs = Xs[:,:3]
  vs = Xs[:,3:]
  Rs += [rs]

ms = np.array(ms)
Rs = np.array(Rs)
Rs.shape

(3, 330, 3)

Ahora, hacemos un doble ciclo for para calcular la energía potencial entre los cuerpos del sistema asi:

$$ U = -\frac{1}{2} ∑_i ∑_{i\neq j} U_{ij} = -\frac{1}{2} ∑_i ∑_{i\neq j} \frac{-Gm_{i}m_{j}} {\vec{r}_{ij}} $$

$$ U =  ∑_i ∑_{i < j} U_{ij} = ∑_i ∑_{i < j} \frac{-Gm_{i}m_{j}} {\vec{r}_{ij}} $$

La energía potencial se puede expesar de las anteriores maneras, las dos expresiones garantizan que no se repitan los vectores $r_{ij} = r_i - r_j$, para este códijo se usó la segunda ecuación, el índice i es menor que j, lo que nos da como resultado las combinaciones de vectores (0,1), (0,2) y (1,2); de esta manera no hay vectores de distancia repetidos entre cuerpos.

In [14]:
U_total = 0

for i in range(3):
  for j in range(3):
    if i < j:
      U = - G * ms[i] * ms[j] / np.linalg.norm(Rs[i]-Rs[j], axis = 1)
      U_total += U

U_mean = U_total.mean()
print("La energía potencial promedio del sistema es:", U_mean)

La energía potencial promedio del sistema es: -3.765195617229033e+35


Ahora calculamos la energía total del sistema, lo que significa sumar la energía cinética promedio y la energía potencial promedio:

$$ E = <K> + <U> $$

In [15]:
E = K_total.mean() + U_total.mean()
print("La energía total del sistema es:", E)

La energía total del sistema es: -1.8830822962240773e+35


Ahora, verificamos si se cumple el teorema del virial:

$$ E + K = 0 $$
$$ 2K + U = 0 ---> K+\frac{1}{2}U =0 $$

In [19]:
print("La energía cinética K es", K_mean)
print("La energía potencial U es", U_mean)
print("La energía total es", E)
print('—'*50)

d_KU = abs(U_mean/2 + K_mean) / abs(K_mean) * 100
print("La mitad promedio de U difiere de K en", d_KU, "%")

d_EK = abs(E + K_mean) / abs(K_mean) * 100
print("E difiere de K en", d_EK, "%")

La energía cinética K es 1.882113321004956e+35
La energía potencial U es -3.765195617229033e+35
La energía total es -1.8830822962240773e+35
——————————————————————————————————————————————————
La mitad promedio de U difiere de K en 0.025741681127998673 %
E difiere de K en 0.051483362255997346 %


Los resultados anteriores permiten observar que el sistema Sol-Júpiter-Saturno está virializado, por lo tanto esta ligado.

## **Sistema de cuerpos: Sol-Júpiter-Saturno-Urano-Neptuno**

Para este caso, se calcula de la misma manera que para el sistema Sol-Júpiter-Saturno la energía cinética y potencial, pero al extraer datos de pymcel, se agrega Urano (con id 799) y Neptuno (con id 899).

In [20]:
K_total = 0

for id in '10', '5', '6', '799', '899':
  # Extraemos los datos
  m = pc.consulta_propiedad(id = id, propiedad = 'masa')
  Xs, ts, Xs_df = pc.consulta_horizons(
        id = id,
        location = '@0',
        epochs =dict(start = '2024-09-23', stop = '2353-09-23', step = '365d')
        )

  # Sacamos las posiciones y velocidades
  Xs = np.array(Xs_df)
  rs = Xs[:,:3]
  vs = Xs[:,3:]

  # Energía cinetica de un cada cuerpo
  K = 1/2 * m * np.linalg.norm(vs, axis=1) ** 2

  K_total += K

K_mean = K_total.mean()
print("La energía cinética promedio del sistema es:", K_mean)

La energía cinética promedio del sistema es: 1.9173517080442617e+35


In [24]:
Rs = []
ms = []
for id in '10', '5', '6', '799', '899':
  # Extraemos los datos
  m = pc.consulta_propiedad(id=id,propiedad='masa')
  ms += [m]

  Xs,ts,Xs_df = pc.consulta_horizons(
      id=id,
      location='@0',
      epochs=dict(start='2024-09-23',stop='2353-09-23',step='365d')
  )

  # Sacamos posiciones y velocidades
  Xs = np.array(Xs_df)
  rs = Xs[:,:3]
  vs = Xs[:,3:]
  Rs += [rs]

ms = np.array(ms)
Rs = np.array(Rs)

In [25]:
U_total = 0

for i in range(5):
  for j in range(5):
    if i < j:
      U = - G * ms[i] * ms[j] / np.linalg.norm(Rs[i]-Rs[j], axis = 1)
      U_total += U

U_mean = U_total.mean()
print("La energía potencial promedio del sistema es:", U_mean)

La energía potencial promedio del sistema es: -3.835649629687023e+35


In [26]:
E = K_total.mean() + U_total.mean()

In [27]:
print("La energía cinética K es", K_mean)
print("La energía potencial U es", U_mean)
print("La energía total es", E)
print('—'*50)

d_KU = abs(U_mean/2 + K_mean) / abs(K_mean) * 100
print("La mitad promedio de U difiere de K en", d_KU, "%")

d_EK = abs(E + K_mean) / abs(K_mean) * 100
print("E difiere de K en", d_EK, "%")

La energía cinética K es 1.9173517080442617e+35
La energía potencial U es -3.835649629687023e+35
La energía total es -1.9182979216427615e+35
——————————————————————————————————————————————————
La mitad promedio de U difiere de K en 0.024675013836270764 %
E difiere de K en 0.04935002767254153 %


El porcentaje que difiere la mitad promedio de U con K y el que difiere E con K es menor en este sistema de 5 cuerpos del sistema solar; lo que significa también que el sistema continua virializado y ligado.